# 1일차 1교시 — 강화학습 소개

**PyTorch로 배우는 강화학습 · 1일차 Tabular-based Methods · 2026-07-27 (월)**

이애본 (Ph.D Aebon) · DreamIT Biz · https://pytorch26.dreamitbiz.com

---

## 🎯 학습목표

- 강화학습이 지도학습·비지도학습과 어떻게 다른지 설명할 수 있다
- 에이전트-환경 상호작용 루프(상태·행동·보상)를 이해한다
- 탐험(Exploration)과 활용(Exploitation)의 트레이드오프를 이해한다

---

# ⚡ 실행 방법 두 가지 — 편한 쪽을 고르세요

### 방법 ① 통째로 한 번에
바로 아래 **[통째로 실행]** 셀 **하나만** 실행하면 끝까지 돕니다.
결과부터 보고 싶으신 분께 권합니다.

### 방법 ② 단계별로 하나씩
그 아래 **[단계별]** 부분을 위에서부터 `Shift + Enter` 로 하나씩 실행하세요.
모두 **5칸**입니다. 한 칸 돌리고 결과 보고 넘어가면 됩니다.

> **이 교시는 혼자 돌아갑니다.** 앞 교시를 먼저 실행하지 않아도 됩니다.
> (앞 교시에서 만든 것을 이 노트북 안에 다시 넣어 뒀습니다 — 사이트의 *이 교시 전체 코드* 와 같은 판입니다.)
> 설치할 것도 없습니다 — 코랩에 다 들어 있습니다.

---

# ① 통째로 한 번에 실행

GitHub 에서 원본을 받아 그대로 돌립니다. 원본이 고쳐지면 자동으로 최신을 받습니다.

In [ ]:
!curl -sL https://raw.githubusercontent.com/aebonlee/pytorch26-lab/main/day1/standalone/01_bandit_epsilon_greedy.py -o 01_bandit_epsilon_greedy.py
!python 01_bandit_epsilon_greedy.py

---

# ② 단계별로 하나씩 실행

이 교시 내용이 **5칸**입니다.
위에서부터 `Shift + Enter`.

> ①을 이미 돌리셨어도 상관없습니다. 처음부터 다시 하는 것과 같습니다.

### 1 / 5 칸

In [ ]:
# ============================================================
# 1일차 1교시 — 강화학습 소개
# 복사해서 그대로 실행하면 됩니다. 고칠 것 없습니다.
# ------------------------------------------------------------
# 이 교시 코드는 앞 교시의 변수·클래스를 이어 씁니다.
# 그래서 이 블록에는 **여기까지 필요한 코드가 전부** 들어 있습니다.
# (수업용 코드만 따로 복사하면 NameError 가 납니다 — 그건 정상입니다.)
# ============================================================

# ── 오늘 이 교시 — 강화학습 소개 ──
import numpy as np                          # 숫자 계산 도구 (파이썬의 계산기)

### 2 / 5 칸

In [ ]:
# ============================================================
# 슬롯머신 10대 중에 좋은 걸 찾기 — 탐험과 활용의 첫 만남
# ------------------------------------------------------------
# 슬롯머신이 10대 있습니다. 각각 나오는 돈의 평균이 다릅니다.
# 그런데 어느 게 좋은지는 ==해봐야 알 수 있습니다.==
#
# 여기서 딜레마가 생깁니다.
#   활용(exploit) : 지금까지 제일 좋았던 걸 계속 당긴다
#   탐험(explore) : 다른 것도 가끔 당겨 본다
#
# 활용만 하면? 처음 운 좋게 나온 것에 갇힙니다.
# 탐험만 하면? 좋은 걸 알면서도 계속 딴 걸 당깁니다.
# ============================================================

np.random.seed(0)                           # 결과를 항상 같게 만든다 (수업용)

n_arms = 10                                 # 슬롯머신 10대
true_means = np.random.normal(0, 1, n_arms) # 각 기계의 '진짜' 평균 (우리는 모른다고 치자)

### 3 / 5 칸

In [ ]:
def run_bandit(epsilon, steps=2000):
    """
    epsilon 확률로 아무거나 당기고, 나머지는 제일 좋아 보이는 걸 당긴다.
    steps 번 당겨 보고 평균 수익을 돌려준다.
    """
    Q = np.zeros(n_arms)        # 각 기계가 얼마나 좋은지 내 '추정치'. 0에서 시작.
    N = np.zeros(n_arms)        # 각 기계를 몇 번 당겼는지 세는 통

    rewards = []                # 매번 받은 돈을 기록

    for t in range(steps):      # steps 번 반복
        # ── ① 어느 기계를 당길지 고른다 ──
        if np.random.rand() < epsilon:      # 0~1 사이 무작위 수가 epsilon 보다 작으면
            a = np.random.randint(n_arms)   #   아무거나 고른다 (탐험)
        else:
            a = np.argmax(Q)                #   추정치가 가장 큰 걸 고른다 (활용)

        # ── ② 실제로 당겨 본다 ──
        r = np.random.normal(true_means[a], 1)   # 진짜 평균 근처에서 값이 나온다
                                                 # 1은 흔들림의 크기 (매번 다르게 나온다)

        # ── ③ 결과를 반영해 추정치를 고친다 ──
        N[a] += 1                            # 이 기계를 한 번 더 당겼다고 기록
        Q[a] += (r - Q[a]) / N[a]            # 추정치를 새 결과 쪽으로 조금 옮긴다
        # 이 한 줄이 '평균 구하기'입니다. 다 모아 뒀다 나누지 않고
        # 나올 때마다 조금씩 옮겨 가는 방식입니다. 강화학습 내내 이 모양이 나옵니다.
        #   새 추정 = 옛 추정 + (실제로 나온 것 - 옛 추정) x 얼마나 반영할지

        rewards.append(r)                    # 받은 돈 기록

    return np.mean(rewards)                  # 평균 수익을 돌려준다

### 4 / 5 칸

In [ ]:
# ── epsilon 을 바꿔 가며 비교 ────────────────────────────
print('epsilon = 아무거나 당겨 볼 확률')
print()

### 5 / 5 칸

In [ ]:
for eps in [0.0, 0.01, 0.1, 0.5]:
    print(f"epsilon={eps:4.2f}  평균 보상 = {run_bandit(eps):.3f}")

print("""
결과 읽는 법
  epsilon = 0.00  탐험을 아예 안 합니다.
                  처음 운 좋게 나온 기계에 갇혀서 더 좋은 걸 영영 못 찾습니다.
  epsilon = 0.01  아주 가끔만 둘러봅니다. 조심스럽습니다.
  epsilon = 0.10  보통 쓰는 값. 대개 여기쯤이 가장 좋습니다.
  epsilon = 0.50  절반을 딴 데 씁니다. 좋은 걸 알면서도 낭비합니다.

→ 너무 안 해봐도 안 되고, 너무 많이 해봐도 안 됩니다.
  이 균형이 강화학습 3일 내내 따라다니는 문제입니다.

  1일차 : 사람이 epsilon 을 정해 준다
  2일차 : 처음엔 크게 나중엔 작게 줄여 나간다
  3일차 : 아예 목표에 넣어서 스스로 조절하게 만든다
""")

---

## 🔧 바꿔 보기
  1) steps 를 200 으로 줄이면? → 탐험할 시간이 부족해 결과가 나빠집니다
  2) steps 를 20000 으로 늘리면? → epsilon 이 작아도 결국 좋은 걸 찾습니다
  3) np.random.seed(0) 의 0을 1, 2 로 바꿔 보세요.
     순위가 바뀔 수도 있습니다 — 한 번의 결과로 판단하면 안 된다는 뜻입니다.

---

## 막히면

- 사이트의 같은 교시를 보세요 — 실행 결과와 해설이 그대로 있습니다.
  https://pytorch26.dreamitbiz.com/#/day/1/1
- 오류가 나면 **[막힐 때]** 메뉴부터.
  https://pytorch26.dreamitbiz.com/#/help

---

*Ph.D Aebon & Claude Code 협작 전자출판 도서 · © 2026 DreamIT Biz*